# Exercise 3 — Film Format Compliance and Automated Conversion (ffprobe + ffmpeg)

## Aim
This notebook implements an automated workflow for the Narbonne Online Film Festival submission process.

The system performs three main tasks:

1. Inspect the technical properties of submitted film files using **ffprobe**.
2. Identify films that do not comply with the required delivery specification and record the problematic fields in a **TXT report**.
3. Automatically convert non-compliant films to the correct format using **ffmpeg**, generating corrected copies with the suffix `_formatOK`.

## Festival Technical Specification (Target Format)

The festival requires all submissions to comply with the following technical parameters:

- **Container:** MP4  
- **Video codec:** H.265 / HEVC  
- **Audio codec:** AAC  
- **Frame rate:** 25 FPS  
- **Aspect ratio:** 16:9  
- **Resolution:** 640 × 360  
- **Video bit rate:** 2–5 Mb/s  
- **Audio bit rate:** ≤ 256 kb/s  
- **Audio channels:** Stereo (2)

These parameters are used later in the notebook to automatically verify whether each submitted film satisfies the festival requirements.

In [ ]:
import os

# Create input and output folders
os.makedirs("exercise3_input", exist_ok=True)
os.makedirs("exercise3_output", exist_ok=True)

print("Folders created successfully!")

## Step 1 — Preparing the working environment

This section prepares the notebook workspace for the film-format validation task.

The code:
- creates an `exercise3_input/` folder for extracted source files
- creates an `exercise3_output/` folder for converted compliant outputs
- checks that the submitted ZIP archive is present in the working directory
- verifies that `ffmpeg` and `ffprobe` are installed and accessible from Python

This setup is necessary before any extraction, metadata inspection, report generation, or conversion can take place.

In [ ]:
import os

zip_path = "exercise3_files_isp.zip"

if not os.path.exists(zip_path):
    raise FileNotFoundError(
        "exercise3_files_isp.zip was not found. Upload it into the same Jupyter folder as the notebook first."
    )

print("ZIP file found:", zip_path)

In [ ]:
import shutil
import subprocess

ffmpeg_path = shutil.which("ffmpeg")
ffprobe_path = shutil.which("ffprobe")

print("ffmpeg path:", ffmpeg_path)
print("ffprobe path:", ffprobe_path)

if ffmpeg_path is None or ffprobe_path is None:
    raise EnvironmentError(
        "ffmpeg and/or ffprobe are not available in this Jupyter environment."
    )

print("\nffmpeg version:")
subprocess.run(["ffmpeg", "-version"], check=True)

print("\nffprobe version:")
subprocess.run(["ffprobe", "-version"], check=True)

## Step 2 — Extracting the submitted film files

The uploaded ZIP archive is extracted into:

`exercise3_input/Exercise3_Files/`

The code first removes any previous extracted copy of the folder to avoid mixing old files with the current submission. It then creates a clean extraction directory and unpacks all submitted film excerpts into it for subsequent inspection.

In [ ]:
import os
import zipfile
import shutil

extract_folder = "exercise3_input/Exercise3_Files"

# Remove old extracted folder if it already exists
if os.path.exists(extract_folder):
    shutil.rmtree(extract_folder)

os.makedirs(extract_folder, exist_ok=True)

with zipfile.ZipFile("exercise3_files_isp.zip", "r") as z:
    z.extractall(extract_folder)

print("ZIP extracted successfully into:", extract_folder)

for root, dirs, files in os.walk(extract_folder):
    for name in files:
        print(os.path.join(root, name))

In [ ]:
import os

video_folder = "exercise3_input/Exercise3_Files"

if not os.path.exists(video_folder):
    raise FileNotFoundError(f"Folder not found: {video_folder}")

videos = os.listdir(video_folder)

print("Films found:")
for v in videos:
    print(v)

## Step 3 — Locating the submitted video files

After extraction, the notebook recursively scans `exercise3_input/Exercise3_Files` and builds a list of all file paths corresponding to the submitted film excerpts.

These paths are then used as the input set for:
- metadata inspection with `ffprobe`
- compliance checking against the festival specification
- TXT report generation
- automatic conversion of non-compliant files with `ffmpeg`

This step ensures that all submitted files are processed systematically rather than manually.

In [ ]:
import glob
import os

video_paths = sorted(glob.glob("exercise3_input/Exercise3_Files/**/*", recursive=True))
video_paths = [p for p in video_paths if os.path.isfile(p)]

print("Total files:", len(video_paths))
for p in video_paths:
    print(p)

## Step 4 — Compliance checking with ffprobe and TXT report generation

Each submitted film is analysed using `ffprobe`, which returns structured metadata about the container and media streams. The notebook compares this metadata against the festival delivery specification.

The validation checks:
- container format
- video codec
- audio codec
- frame rate
- resolution
- aspect ratio
- video bit rate
- audio bit rate
- stereo audio requirement (2 channels)

If a file fails one or more checks, it is marked as **NOT OK** and the failing fields are recorded.

### Required TXT output

The notebook then writes a brief TXT report (`exercise3_report.txt`) listing:
- the films that do not comply with the festival specification
- the problematic fields for each non-compliant file
- a short summary of compliant and non-compliant files

This directly addresses the coursework requirement to identify which films do not respect the required format and specify the problematic fields.

In [ ]:
import os, json, subprocess, math, pathlib

if len(video_paths) == 0:
    raise ValueError(
        "No video files were found. Make sure exercise3_files_isp.zip was uploaded and extracted correctly."
    )

# ---------- Festival specification (from coursework) ----------
SPEC = {
    "container": "mp4",
    "vcodec": ["hevc", "h265"],
    "acodec": ["aac"],
    "fps": 25.0,
    "width": 640,
    "height": 360,
    "aspect_ratio": (16, 9),
    "vbitrate_min": 2_000_000,
    "vbitrate_max": 5_000_000,
    "abitrate_max": 256_000,
    "audio_channels": 2
}

REPORT_PATH = "exercise3_report.txt"

def ffprobe_json(path):
    cmd = [
        "ffprobe", "-v", "error",
        "-print_format", "json",
        "-show_format", "-show_streams",
        path
    ]
    out = subprocess.check_output(cmd)
    return json.loads(out.decode("utf-8", errors="replace"))

def parse_ratio(r):
    try:
        a, b = r.split(":")
        return (int(a), int(b))
    except:
        return None

def parse_fps(rf):
    try:
        n, d = rf.split("/")
        n, d = float(n), float(d)
        return 0.0 if d == 0 else n / d
    except:
        return None

def get_streams(meta):
    v = None
    a = None
    for s in meta.get("streams", []):
        if s.get("codec_type") == "video" and v is None:
            v = s
        elif s.get("codec_type") == "audio" and a is None:
            a = s
    return v, a

def check_file(path):
    meta = ffprobe_json(path)
    fmt = meta.get("format", {})
    v, a = get_streams(meta)

    problems = []

    # container
    container_ok = SPEC["container"] in (fmt.get("format_name", "") or "")
    if not container_ok:
        problems.append("container")

    # video stream presence
    if v is None:
        problems += ["video_codec", "frame_rate", "resolution", "aspect_ratio", "video_bitrate"]
    else:
        if (v.get("codec_name") or "").lower() not in SPEC["vcodec"]:
            problems.append("video_codec")

        fps = parse_fps(v.get("r_frame_rate", "0/0"))
        if fps is None or abs(fps - SPEC["fps"]) > 0.01:
            problems.append("frame_rate")

        if int(v.get("width", 0)) != SPEC["width"] or int(v.get("height", 0)) != SPEC["height"]:
            problems.append("resolution")

        dar = v.get("display_aspect_ratio")
        if dar:
            ar = parse_ratio(dar)
            if ar != SPEC["aspect_ratio"]:
                problems.append("aspect_ratio")
        else:
            w = int(v.get("width", 0))
            h = int(v.get("height", 1) or 1)
            if not math.isclose(w / h, SPEC["aspect_ratio"][0] / SPEC["aspect_ratio"][1], rel_tol=1e-3):
                problems.append("aspect_ratio")

        vbr = v.get("bit_rate") or meta.get("format", {}).get("bit_rate")
        try:
            vbr = int(vbr)
            if not (SPEC["vbitrate_min"] <= vbr <= SPEC["vbitrate_max"]):
                problems.append("video_bitrate")
        except:
            problems.append("video_bitrate")

    # audio stream presence
    if a is None:
        problems += ["audio_codec", "audio_bitrate", "audio_channels"]
    else:
        if (a.get("codec_name") or "").lower() not in SPEC["acodec"]:
            problems.append("audio_codec")

        try:
            abr = int(a.get("bit_rate"))
            if abr > SPEC["abitrate_max"]:
                problems.append("audio_bitrate")
        except:
            problems.append("audio_bitrate")

        if int(a.get("channels", 0)) != SPEC["audio_channels"]:
            problems.append("audio_channels")

    return {
        "path": path,
        "filename": pathlib.Path(path).name,
        "ok": len(problems) == 0,
        "problems": problems
    }

results = []

for p in video_paths:
    try:
        results.append(check_file(p))
    except Exception as e:
        results.append({
            "path": p,
            "filename": pathlib.Path(p).name,
            "ok": False,
            "problems": [f"unreadable_file: {str(e)}"]
        })

ok_files = [r for r in results if r["ok"]]
bad_files = [r for r in results if not r["ok"]]

with open(REPORT_PATH, "w", encoding="utf-8") as f:
    f.write("Exercise 3 – Film Verification Report\n")
    f.write("=====================================\n\n")
    f.write(f"Total files checked: {len(results)}\n")
    f.write(f"OK files: {len(ok_files)}\n")
    f.write(f"NOT OK files: {len(bad_files)}\n\n")

    if bad_files:
        f.write("Files that do NOT respect the required format:\n")
        for r in bad_files:
            f.write(f"- {r['filename']}: {', '.join(r['problems'])}\n")
    else:
        f.write("All files are OK.\n")

print("TXT report written to:", REPORT_PATH)
print("OK files:", len(ok_files))
print("NOT OK files:", len(bad_files))

for r in bad_files:
    print(r["filename"], "->", r["problems"])

## Step 5 — Automatic conversion of non-compliant films

Any film identified as **NOT OK** is passed to `ffmpeg` for conversion into a new MP4 file.

The conversion command targets the festival specification by enforcing:
- MP4 container
- HEVC / H.265 video
- AAC audio
- 25 fps
- 640 × 360 resolution
- stereo audio
- output filename ending in `_formatOK.mp4`

The converted files are written to `exercise3_output/`.

At this stage, the notebook produces standardised outputs for all non-compliant inputs. A later verification step is then used to confirm whether each converted file satisfies all required properties.

## Step 6 — Note on practical encoding verification

AAC bitrate values reported by media tools can sometimes vary slightly because of encoder behaviour and metadata reporting. For that reason, the notebook performs a final verification step after conversion rather than assuming that every generated file is fully compliant immediately after encoding.

This is good practice in automated media processing, because the conversion stage and the validation stage should be treated as separate steps.

In [ ]:
import os, subprocess, pathlib

converted = []

for r in bad_files:
    in_path = r["path"]
    stem = pathlib.Path(in_path).stem
    out_path = os.path.join("exercise3_output", f"{stem}_formatOK.mp4")

    cmd = [
        "ffmpeg", "-y",
        "-i", in_path,
        "-c:v", "libx265",
        "-r", "25",
        "-vf", "scale=640:360",
        "-b:v", "3000k",
        "-c:a", "aac",
        "-ac", "2",
        "-b:a", "256k",
        out_path
    ]

    print("Converting:", pathlib.Path(in_path).name)
    subprocess.run(cmd, check=True)
    converted.append(out_path)

print("Converted:", len(converted))
for c in converted:
    print(c)

## Step 7 — Preparing for final output checks

This section handles any additional processing logic applied to converted files before the final validation stage. Its purpose is to ensure that the notebook only carries forward existing converted outputs for final inspection.

In [ ]:
import os, subprocess, pathlib

reconverted = []

for path in converted:
    if not os.path.exists(path):
        continue

    stem = pathlib.Path(path).stem
    out_path = os.path.join("exercise3_output", f"{stem}.mp4")

    if os.path.abspath(path) == os.path.abspath(out_path):
        continue

    cmd = [
        "ffmpeg", "-y",
        "-i", path,
        "-c:v", "copy",
        "-c:a", "aac",
        "-b:a", "256k",
        out_path
    ]

    print("Reprocessing audio for:", pathlib.Path(path).name)
    subprocess.run(cmd, check=True)
    reconverted.append(out_path)

print("Reprocessed:", len(reconverted))
for c in reconverted:
    print(c)

## Step 8 — Final verification of converted outputs

As a final quality-control stage, the generated MP4 outputs are checked again using the same compliance function applied to the original submissions.

This confirms whether the converted `_formatOK` files satisfy the required delivery specification and provides a clear end-point for the automated workflow.

In [ ]:
final_outputs = sorted([
    os.path.join("exercise3_output", f)
    for f in os.listdir("exercise3_output")
    if f.lower().endswith(".mp4")
])

print("Final output files:", len(final_outputs))

for p in final_outputs:
    r = check_file(p)
    print(
        pathlib.Path(p).name,
        "->",
        "OK" if r["ok"] else f"NOT OK: {', '.join(r['problems'])}"
    )